# Scalable indicator calculations with Dask

This tutorial demonstrates how to run  indicator calculations on larger datasets using **Dask** for lazy evaluation and parallel computation.

> **Key Concept**: Before continuing with this hands-on tutorial, we strongly recommend reading the concept page on [Scalability and performance](../concepts/scalability_performance.rst), which explains Dask chunking principles, spatial vs. temporal chunking, task vs. peer-to-peer rechunking engines, and worker memory management.

Datasets loaded via  (e.g. from a GRIB/NetCDF ) can be passed directly to indicators or converted to lazy chunked arrays.


In [ ]:
import dask.array as da
import numpy as np
import pandas as pd
import xarray as xr

import earthkit.climate as ekc

## 1. Creating a chunked lazy dataset

When working with large climate datasets (e.g. ERA5 or CMIP6), data arrays are loaded as **dask-backed DataArrays**.
Here we create a synthetic 3D spatial-temporal temperature dataset (, , ) backed by Dask arrays.


In [ ]:
# 10 years of daily data on a 20x20 spatial grid
dates = pd.date_range("2010-01-01", "2019-12-31", freq="D")
lats = np.linspace(35, 60, 20)
lons = np.linspace(-10, 30, 20)

# Generate lazy Dask array for maximum temperature
dask_data = da.random.normal(285.0, 10.0, size=(len(dates), len(lats), len(lons)), chunks=(365, 10, 10))

tasmax = xr.DataArray(
    dask_data,
    coords={"time": dates, "lat": lats, "lon": lons},
    dims=["time", "lat", "lon"],
    name="tasmax",
    attrs={"units": "K", "standard_name": "air_temperature"},
)

print("Lazy DataArray structure:")
print(tasmax)

## 2. Computing indicators lazily

When passing a dask-backed  to  indicator functions, the index calculation is evaluated **lazily**.
The function returns a new DataArray containing a Dask computational task graph without performing heavy numerical computation immediately.


In [ ]:
# Compute annual hot days (> 300 K) on chunked data
hot_days_lazy = ekc.indicators.tx_days_above(tasmax, thresh="300 K", freq="YS")

print("Lazy indicator output (notice it is a Dask array):")
print(hot_days_lazy)

## 3. Triggering computation and saving results

To execute the task graph across available CPU threads, call  or write directly to a NetCDF/Zarr store using .


In [ ]:
# Trigger compute
hot_days_computed = hot_days_lazy.compute()

print("Computed indicator result shape and values:")
print(hot_days_computed)

## 4. Groupby considerations for daily climatologies and percentiles

Operations involving  or percentile thresholds require contiguous time series for each spatial point.

* **Best practice**: Ensure the  dimension is unchunked (or chunked in multi-year blocks) before computing daily percentiles or climatologies.
* **Rechunking**: If your disk data is time-sliced (e.g. 1 day per chunk), rechunk spatially: .


In [ ]:
# Rechunk to ensure contiguous time axis per spatial chunk
tasmax_rechunked = tasmax.chunk({"time": -1, "lat": 5, "lon": 5})

# Compute rolling percentiles on rechunked dataset
per90_lazy = ekc.utils.climatology.rolling_percentiles(tasmax_rechunked, p=90, window_width=5)
print("Rolling percentiles lazy task graph created successfully:")
print(per90_lazy)